In [4]:
# 代码作用：读取MES、LIMS数据汇总为 002炼糖期分蜜指标统计报表v1
# 核心逻辑：全局累计合格率（按月度、班次独立累计）
# 2026/04/02
import pandas as pd
from datetime import datetime
from sqlalchemy import create_engine
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType, DecimalType, DateType,DoubleType
# 关键：明确导入Spark函数，避免和Python内置函数混淆
from pyspark.sql.functions import (
    current_timestamp, split, lit, col, row_number, sum as spark_sum, 
    round as spark_round, avg, when, to_date, count, coalesce
)
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.utils import AnalysisException
import warnings
warnings.filterwarnings("ignore")

print("Spark 启动中...")
spark = SparkSession.builder \
    .appName("lims_dwd") \
    .enableHiveSupport() \
    .getOrCreate()



# ===================== 1. 读取LIMS样品分析 R3糖 数据 =====================
lims_r3_sql = """
WITH raw_data AS (
    SELECT 
        ord_accept_org_code AS company,
        ord_crushing_season AS crushing_season,
        ord_accept_date AS record_date,
        ord_class AS work_shift,
        ord_sample_name,
        ord_order_no,
        test_name1,                     -- 补上逗号
        CASE 
            WHEN test_test_value REGEXP '^[0-9.]+$' THEN CAST(test_test_value AS DECIMAL(10,2))
            ELSE NULL
        END AS test_value
    FROM dwd.dwd_lims_sample_analysis_f_1h
    WHERE 
        ord_proc_st != 'PROC_ZF' 
        AND ord_accept_org_code = 'FNR'
        AND ord_productioncategory = '炼糖生产'
        AND ord_sample_name LIKE '%R3糖浆%'                     
        AND test_name1 IN ('锤度', '色值')                          
)
SELECT 
    company,
    crushing_season,
    record_date,
    work_shift,
    ord_sample_name,
    ord_order_no,
    MAX(CASE WHEN test_name1 = '锤度' THEN test_value END) AS r3_brix,
    MAX(CASE WHEN test_name1 = '色值' THEN test_value END) AS r3_color_value
FROM raw_data
GROUP BY company, crushing_season, record_date, work_shift, ord_sample_name, ord_order_no
HAVING MAX(CASE WHEN test_name1 = '锤度' THEN test_value END) IS NOT NULL 
    OR MAX(CASE WHEN test_name1 = '色值' THEN test_value END) IS NOT NULL;
"""
lims_r3_df = spark.sql(lims_r3_sql)

# ===================== 2. 读取LIMS样品分析 炼糖桔水 数据 =====================
lims_js_sql = """
WITH raw_data AS (
    SELECT 
        ord_accept_org_code AS company,
        ord_crushing_season AS crushing_season,
        ord_accept_date AS record_date,
        ord_class AS work_shift,
        ord_sample_name,
        test_name1,
        ord_up_ver,
        CASE 
            WHEN test_test_value REGEXP '^[0-9.]+$' THEN CAST(test_test_value AS DECIMAL(10,2))
            ELSE NULL
        END AS test_value
    FROM dwd.dwd_lims_sample_analysis_f_1h
    WHERE 
        ord_proc_st != 'PROC_ZF' 
        AND ord_accept_org_code = 'FNR'
        AND ord_productioncategory = '炼糖生产'
        AND ord_sample_name LIKE '%炼糖桔水%'
        AND test_name1 IN ('锤度', '重力纯度', '产量')
),
paste_b_data AS (
    SELECT 
        company,
        crushing_season,
        record_date AS paste_record_date,
        work_shift AS paste_work_shift,
        MIN(ord_up_ver) AS min_ord_up_ver,
        MAX(CASE WHEN test_name1 = '锤度' THEN test_value END) AS molasses_brix,
        MAX(CASE WHEN test_name1 = '重力纯度' THEN test_value END) AS gravity_purity,
        MAX(CASE WHEN test_name1 = '产量' THEN test_value END) AS yield
    FROM raw_data
    WHERE ord_sample_name LIKE '%炼糖桔水%'
    GROUP BY company, crushing_season, record_date, work_shift
),
ranked_paste_data AS (
    SELECT 
        *,
        ROW_NUMBER() OVER (
            PARTITION BY company, crushing_season, paste_record_date, paste_work_shift
            ORDER BY min_ord_up_ver ASC
        ) AS rn
    FROM paste_b_data
)
SELECT 
    company,
    crushing_season,
    paste_record_date,
    paste_work_shift,
    molasses_brix,
    gravity_purity,
    yield
FROM ranked_paste_data
WHERE rn = 1
  AND (
        (CASE WHEN molasses_brix IS NOT NULL THEN 1 ELSE 0 END) +
        (CASE WHEN gravity_purity IS NOT NULL THEN 1 ELSE 0 END) +
        (CASE WHEN yield IS NOT NULL THEN 1 ELSE 0 END)
      ) >= 2
"""
lims_js_df = spark.sql(lims_js_sql)

# ===================== 3. 读取LIMS样品分析 洄溶糖 数据 =====================
lims_hrt_sql = """
WITH raw_data AS (
    SELECT 
        ord_accept_org_code AS company,
        ord_crushing_season AS crushing_season,
        ord_accept_date AS record_date,
        ord_class AS work_shift,
        ord_sample_name,
        ord_id,
        test_name1,
        CASE 
            WHEN test_test_value REGEXP '^[0-9.]+$' THEN CAST(test_test_value AS DECIMAL(10,2))
            ELSE NULL
        END AS test_value
    FROM dwd.dwd_lims_sample_analysis_f_1h
    WHERE 
        ord_proc_st != 'PROC_ZF' 
        AND ord_accept_org_code = 'FNR'
        AND ord_productioncategory = '炼糖生产'
        AND ord_sample_name = '洄溶糖'
        AND test_name1 = '重量'
)
SELECT 
    company,
    crushing_season,
    record_date,
    work_shift,
    ord_sample_name,
    ord_id,
    MAX(CASE WHEN test_name1 = '重量' THEN test_value END) AS remelt_syrup_yield
FROM raw_data
GROUP BY company, crushing_season, record_date, work_shift, ord_sample_name, ord_id
HAVING MAX(CASE WHEN test_name1 = '重量' THEN test_value END) IS NOT NULL
ORDER BY record_date DESC
"""
lims_hrt_df = spark.sql(lims_hrt_sql)

# ===================== 3. 读取LIMS 精糖指标合格标准数据 =====================
lims_jtzb_sql = """
SELECT 
*
FROM dwd.dwd_lims_sugar_quality_standard_f_1h
WHERE org_code = 'FNR'
  AND production_category = '炼糖生产'
  AND material_name IN ('R3糖浆','炼糖桔水')
  AND dismonth_date = (
      SELECT MAX(dismonth_date)
      FROM dwd.dwd_lims_sugar_quality_standard_f_1h
      WHERE org_code = 'FNR'
        AND production_category = '炼糖生产'
        AND material_name IN ('R3糖浆','炼糖桔水')
  )
;
"""
lims_jtzb_df = spark.sql(lims_jtzb_sql)

# ===================== 4. 读取LIMS集团报 废蜜产率目标值 =====================
lims_mb_sql = """
SELECT 
    regexp_extract(type, '([0-9]{2}/[0-9]{2})', 1) AS season_year,
    'FNR' AS factory_code,
    number AS molasses_yield_rate_with_r3
FROM dwd.dwd_mes_sugar_quality_standard_f_1h 
WHERE description LIKE '%糖蜜产率%' 
    AND type LIKE '%合计%' 
    AND subtype = '炼糖期合计' 
    AND number != ''
"""
lims_mb_df = spark.sql(lims_mb_sql)
lims_mb_df.show()

Spark 启动中...
+-----------+------------+---------------------------+
|season_year|factory_code|molasses_yield_rate_with_r3|
+-----------+------------+---------------------------+
|      24/25|         FNR|                       1.65|
|      25/26|         FNR|                       1.65|
+-----------+------------+---------------------------+



In [5]:
# ===================== 后续代码：榨季累计指标（用于柱状图，班次包含“目标”） =====================
from pyspark.sql.functions import col, when, count, sum as spark_sum, round as spark_round, lit
from pyspark.sql.functions import when as when_

print("开始计算榨季累计指标（班次含甲/乙/丙/丁/累计/目标）...")

# -------------------------------------------------------------------
# 辅助函数：统一桔水和洄溶糖的班次、日期字段名
# -------------------------------------------------------------------
def normalize_shift_date(df, shift_col, date_col):
    return df.withColumnRenamed(shift_col, "work_shift") \
             .withColumnRenamed(date_col, "record_date")

js_df = normalize_shift_date(lims_js_df, "paste_work_shift", "paste_record_date")
hrt_df = lims_hrt_df

# -------------------------------------------------------------------
# 1. R3糖浆锤度累计合格率
# -------------------------------------------------------------------
brix_std = lims_jtzb_df.filter(
    (col("material_name") == "R3糖浆") & (col("item_name") == "锤度")
).select("lower_limit", "upper_limit").first()
BRIX_LOWER, BRIX_UPPER = (brix_std["lower_limit"], brix_std["upper_limit"]) if brix_std else (58.0, 63.0)
print(f"R3锤度标准: {BRIX_LOWER} ~ {BRIX_UPPER}")

r3_brix = lims_r3_df.filter(col("r3_brix").isNotNull()) \
    .withColumn("is_qualified", when((col("r3_brix") >= BRIX_LOWER) & (col("r3_brix") <= BRIX_UPPER), 1).otherwise(0))

brix_shift = r3_brix.groupBy("company", "crushing_season", "work_shift") \
    .agg(spark_sum("is_qualified").alias("qualified_cnt"), count("r3_brix").alias("total_cnt")) \
    .withColumn("value", spark_round(col("qualified_cnt") / col("total_cnt"), 4)) \
    .select("company", "crushing_season", lit("R3糖浆锤度").alias("indicator"), 
            col("work_shift").alias("shift"), "value")

brix_dept = r3_brix.groupBy("company", "crushing_season") \
    .agg(spark_sum("is_qualified").alias("qualified_cnt"), count("r3_brix").alias("total_cnt")) \
    .withColumn("value", spark_round(col("qualified_cnt") / col("total_cnt"), 4)) \
    .select("company", "crushing_season", lit("R3糖浆锤度").alias("indicator"), 
            lit("累计").alias("shift"), "value")

r3_brix_res = brix_shift.union(brix_dept)

# -------------------------------------------------------------------
# 2. R3糖浆色值累计合格率
# -------------------------------------------------------------------
color_std = lims_jtzb_df.filter(
    (col("material_name") == "R3糖浆") & (col("item_name") == "色值")
).select("upper_limit").first()
COLOR_UPPER = color_std["upper_limit"] if color_std and color_std["upper_limit"] is not None else 500.0
print(f"R3色值标准: ≤ {COLOR_UPPER}")

r3_color = lims_r3_df.filter(col("r3_color_value").isNotNull()) \
    .withColumn("is_qualified", when(col("r3_color_value") <= COLOR_UPPER, 1).otherwise(0))

color_shift = r3_color.groupBy("company", "crushing_season", "work_shift") \
    .agg(spark_sum("is_qualified").alias("qualified_cnt"), count("r3_color_value").alias("total_cnt")) \
    .withColumn("value", spark_round(col("qualified_cnt") / col("total_cnt"), 4)) \
    .select("company", "crushing_season", lit("R3糖浆色值").alias("indicator"), 
            col("work_shift").alias("shift"), "value")

color_dept = r3_color.groupBy("company", "crushing_season") \
    .agg(spark_sum("is_qualified").alias("qualified_cnt"), count("r3_color_value").alias("total_cnt")) \
    .withColumn("value", spark_round(col("qualified_cnt") / col("total_cnt"), 4)) \
    .select("company", "crushing_season", lit("R3糖浆色值").alias("indicator"), 
            lit("累计").alias("shift"), "value")

r3_color_res = color_shift.union(color_dept)

# -------------------------------------------------------------------
# 3. 桔水锤度累计合格率
# -------------------------------------------------------------------
js_brix_std = lims_jtzb_df.filter(
    (col("material_name") == "炼糖桔水") & (col("item_name") == "锤度")
).select("lower_limit", "upper_limit").first()
JS_BRIX_LOWER, JS_BRIX_UPPER = (js_brix_std["lower_limit"], js_brix_std["upper_limit"]) if js_brix_std else (86.0, 90.0)
print(f"桔水锤度标准: {JS_BRIX_LOWER} ~ {JS_BRIX_UPPER}")

js_brix = js_df.filter(col("molasses_brix").isNotNull()) \
    .withColumn("is_qualified", when((col("molasses_brix") >= JS_BRIX_LOWER) & (col("molasses_brix") <= JS_BRIX_UPPER), 1).otherwise(0))

js_brix_shift = js_brix.groupBy("company", "crushing_season", "work_shift") \
    .agg(spark_sum("is_qualified").alias("qualified_cnt"), count("molasses_brix").alias("total_cnt")) \
    .withColumn("value", spark_round(col("qualified_cnt") / col("total_cnt"), 4)) \
    .select("company", "crushing_season", lit("桔水锤度合格率").alias("indicator"), 
            col("work_shift").alias("shift"), "value")

js_brix_dept = js_brix.groupBy("company", "crushing_season") \
    .agg(spark_sum("is_qualified").alias("qualified_cnt"), count("molasses_brix").alias("total_cnt")) \
    .withColumn("value", spark_round(col("qualified_cnt") / col("total_cnt"), 4)) \
    .select("company", "crushing_season", lit("桔水锤度合格率").alias("indicator"), 
            lit("累计").alias("shift"), "value")

js_brix_res = js_brix_shift.union(js_brix_dept)

# -------------------------------------------------------------------
# 4. 桔水重力纯度累计合格率
# -------------------------------------------------------------------
js_purity_std = lims_jtzb_df.filter(
    (col("material_name") == "炼糖桔水") & (col("item_name") == "重力纯度")
).select("upper_limit").first()
JS_PURITY_UPPER = js_purity_std["upper_limit"] if js_purity_std and js_purity_std["upper_limit"] is not None else 57.0
print(f"桔水重力纯度标准: ≤ {JS_PURITY_UPPER}")

js_purity = js_df.filter(col("gravity_purity").isNotNull()) \
    .withColumn("is_qualified", when(col("gravity_purity") <= JS_PURITY_UPPER, 1).otherwise(0))

js_purity_shift = js_purity.groupBy("company", "crushing_season", "work_shift") \
    .agg(spark_sum("is_qualified").alias("qualified_cnt"), count("gravity_purity").alias("total_cnt")) \
    .withColumn("value", spark_round(col("qualified_cnt") / col("total_cnt"), 4)) \
    .select("company", "crushing_season", lit("桔水重力纯度合格率").alias("indicator"), 
            col("work_shift").alias("shift"), "value")

js_purity_dept = js_purity.groupBy("company", "crushing_season") \
    .agg(spark_sum("is_qualified").alias("qualified_cnt"), count("gravity_purity").alias("total_cnt")) \
    .withColumn("value", spark_round(col("qualified_cnt") / col("total_cnt"), 4)) \
    .select("company", "crushing_season", lit("桔水重力纯度合格率").alias("indicator"), 
            lit("累计").alias("shift"), "value")

js_purity_res = js_purity_shift.union(js_purity_dept)

# -------------------------------------------------------------------
# 5. 废糖蜜产率累计（左连接：洄溶糖为主表，缺失桔水产率填 0）
# -------------------------------------------------------------------
# 按日班次粒度预聚合桔水表的 yield
js_agg = js_df.select("company", "crushing_season", "record_date", "work_shift", "yield") \
    .groupBy("company", "crushing_season", "record_date", "work_shift") \
    .agg(spark_sum("yield").alias("yield"))

# 按日班次粒度预聚合洄溶糖表的 remelt_syrup_yield
hrt_agg = hrt_df.select("company", "crushing_season", "record_date", "work_shift", "remelt_syrup_yield") \
    .groupBy("company", "crushing_season", "record_date", "work_shift") \
    .agg(spark_sum("remelt_syrup_yield").alias("remelt_syrup_yield"))

# 左连接：以洄溶糖为主表，桔水数据缺失时 yield 填 0
joined = hrt_agg.join(js_agg, on=["company", "crushing_season", "record_date", "work_shift"], how="left") \
    .fillna({"yield": 0})   # 连接不上的桔水 yield 填 0

# 后续按班次、累计分组求和，并处理 remelt = 0 的情况（产率置 0）
# 注意：不再过滤 remelt_syrup_yield != 0，保留所有班次
yield_shift = joined.groupBy("company", "crushing_season", "work_shift") \
    .agg(spark_sum("yield").alias("sum_yield"), spark_sum("remelt_syrup_yield").alias("sum_remelt")) \
    .withColumn("value", 
                when(col("sum_remelt") == 0, lit(0.0))
                .otherwise(spark_round(col("sum_yield") / col("sum_remelt")*100, 4))) \
    .select("company", "crushing_season", lit("废糖蜜产率").alias("indicator"), 
            col("work_shift").alias("shift"), "value")

yield_dept = joined.groupBy("company", "crushing_season") \
    .agg(spark_sum("yield").alias("sum_yield"), spark_sum("remelt_syrup_yield").alias("sum_remelt")) \
    .withColumn("value", 
                when(col("sum_remelt") == 0, lit(0.0))
                .otherwise(spark_round(col("sum_yield") / col("sum_remelt")*100, 4))) \
    .select("company", "crushing_season", lit("废糖蜜产率").alias("indicator"), 
            lit("累计").alias("shift"), "value")

js_yield_res = yield_shift.union(yield_dept)

# -------------------------------------------------------------------
# 6. 合并所有指标，并添加“目标”班次行（废糖蜜产率目标值从 lims_mb_df 动态获取）
# -------------------------------------------------------------------
final_df = r3_brix_res.union(r3_color_res) \
                      .union(js_brix_res) \
                      .union(js_purity_res) \
                      .union(js_yield_res)

# 准备废糖蜜产率目标值映射表（company, crushing_season -> target_value）
molasses_target_df = lims_mb_df.select(
    col("factory_code").alias("company"),
    col("season_year").alias("crushing_season"),
    col("molasses_yield_rate_with_r3").cast("double").alias("target_molasses_yield")
)

# 生成所有 (company, crushing_season, indicator) 组合（作为目标行的基础）
base_target_rows = final_df.select("company", "crushing_season", "indicator").distinct()

# 废糖蜜产率的目标行：通过左连接获取动态目标值，若缺失则默认 0.07
molasses_target_rows = base_target_rows.filter(col("indicator") == "废糖蜜产率") \
    .join(molasses_target_df, on=["company", "crushing_season"], how="left") \
    .withColumn("value", coalesce(col("target_molasses_yield"), lit(0.07))) \
    .select("company", "crushing_season", "indicator", lit("目标").alias("shift"), "value")

# 其他三个指标的目标行（固定 0.95）
other_target_rows = base_target_rows.filter(col("indicator") != "废糖蜜产率") \
    .withColumn("value", lit(0.95)) \
    .select("company", "crushing_season", "indicator", lit("目标").alias("shift"), "value")

# 合并目标行
target_rows = molasses_target_rows.union(other_target_rows)

# 将目标行添加到最终结果中
final_df_with_target = final_df.union(target_rows)

# 班次排序（甲、乙、丙、丁、累计、目标）
sorted_df = final_df_with_target.withColumn("shift_order",
    when_(col("shift") == "甲班", 1)
    .when(col("shift") == "乙班", 2)
    .when(col("shift") == "丙班", 3)
    .when(col("shift") == "丁班", 4)
    .when(col("shift") == "累计", 5)
    .when(col("shift") == "目标", 6)
    .otherwise(7)
).orderBy("company", "crushing_season", "indicator", "shift_order") \
 .drop("shift_order")

print("榨季累计指标计算完成（班次包含甲/乙/丙/丁/累计/目标）:")
sorted_df.show(200, truncate=False)

# ===================== 写入 Hive 表 =====================
table_name = "app.app_lims_refining_indicators_bar_f_1h"
print(f"正在写入表 {table_name} ...")

spark.sql(f"DROP TABLE IF EXISTS {table_name}")
sorted_df.write.mode("overwrite") \
    .format("parquet") \
    .saveAsTable(table_name)

print(f"数据已成功写入表 {table_name}")
print("作业完成。")

开始计算榨季累计指标（班次含甲/乙/丙/丁/累计/目标）...
R3锤度标准: 58.00 ~ 63.00
R3色值标准: ≤ 500.00
桔水锤度标准: 86.00 ~ 90.00
桔水重力纯度标准: ≤ 57.00
榨季累计指标计算完成（班次包含甲/乙/丙/丁/累计/目标）:
+-------+---------------+------------------+-----+------+
|company|crushing_season|indicator         |shift|value |
+-------+---------------+------------------+-----+------+
|FNR    |22/23          |R3糖浆色值        |甲班 |0.9808|
|FNR    |22/23          |R3糖浆色值        |乙班 |0.9792|
|FNR    |22/23          |R3糖浆色值        |丙班 |0.9796|
|FNR    |22/23          |R3糖浆色值        |丁班 |0.9912|
|FNR    |22/23          |R3糖浆色值        |累计 |0.983 |
|FNR    |22/23          |R3糖浆色值        |目标 |0.95  |
|FNR    |22/23          |R3糖浆锤度        |甲班 |0.9746|
|FNR    |22/23          |R3糖浆锤度        |乙班 |0.9375|
|FNR    |22/23          |R3糖浆锤度        |丙班 |0.9592|
|FNR    |22/23          |R3糖浆锤度        |丁班 |0.9292|
|FNR    |22/23          |R3糖浆锤度        |累计 |0.9544|
|FNR    |22/23          |R3糖浆锤度        |目标 |0.95  |
|FNR    |22/23          |废糖蜜产率        |甲班 |0.5016|
|FNR    |

In [6]:
# ===================== 导出 joined 数据到 Excel（用于验证废糖蜜产率） =====================
# 注意：确保环境中已安装 pandas 和 openpyxl，可通过 !pip install pandas openpyxl 安装
import pandas as pd

# 将 joined 转换为 Pandas DataFrame（如果数据量过大，建议先采样或按条件过滤）
joined_pd = joined.select(
    "company", "crushing_season", "record_date", "work_shift", 
    "yield", "remelt_syrup_yield"
).toPandas()

# 导出到 Excel 文件（请修改为您有写权限的路径，例如 HDFS 路径需用本地路径）
output_path = "molasses_yield_validation.xlsx"
joined_pd.to_excel(output_path, index=False, engine="openpyxl")
print(f"已导出 joined 数据到 {output_path}，共 {len(joined_pd)} 行")

# 可选：同时导出汇总结果 js_yield_res 以便对照
yield_res_pd = js_yield_res.toPandas()
yield_res_pd.to_excel("yield_res_validation.xlsx", index=False)

已导出 joined 数据到 molasses_yield_validation.xlsx，共 3093 行
